# WG FA — Impact des biais systématiques sur les features

**Collaboration B · WG FA (Feature Analysis) × WG SYST · dataset HiggsML (H→ττ)**

Ce notebook mesure l'**impact des biais systématiques** sur la distribution de **chaque** feature.

- Trois biais cinématiques décalent les variables : **TES** (échelle d'énergie du tau, sur `PRI_had_pt`), **JES** (échelle d'énergie des jets) et **soft MET** (terme mou sur la MET). L'effet se **propage** ensuite aux variables dérivées (masses, sommes de pt…) au recalcul (`postprocess`).
- On compare le **nominal** à ses variations, feature par feature, en deux temps :
  1. **superposition** nominal vs ±3 % par variable (grilles PRI / DER, illustrées pour le **TES**) ;
  2. **classement de sensibilité** : un graphe en bâtons par biais (**TES**, **JES**, **soft MET**), avec un score unique par variable.

> Comparaison « à postprocess égal » : le nominal passe lui aussi par `systematics()`, donc la seule différence entre les courbes est le biais lui-même.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

%matplotlib inline
plt.rcParams["figure.dpi"] = 100

## 1. Chargement du dataset (intégralité, via le starting kit)

In [ ]:
DATA_ROOT = None
for base in [Path.cwd(), *Path.cwd().parents]:
    for cand in (base, base / "Higgs_collaborations-main"):
        if (cand / "blackSwan_data").is_dir():
            DATA_ROOT = cand
            break
    if DATA_ROOT:
        break
assert DATA_ROOT is not None, "Dossier 'blackSwan_data' introuvable."
os.chdir(DATA_ROOT)

from HiggsML.datasets import download_dataset

data = download_dataset("blackSwan_data")
data.test_size = 0          # charger l'intégralité du parquet (2 000 000 d'événements)
data.load_train_set()
df = data.get_train_set()
print("Données chargées :", df.shape[0], "événements")

## 2. Application des biais systématiques

`systematics(df, …)` modifie les primaires, propage l'effet (MET, variables dérivées) puis applique le `postprocess` (recalcul des DER_*, seuils en pt qui suppriment des événements). Les trois biais cinématiques :

- **TES** (`tes=f`) : multiplie `PRI_had_pt` (l'impulsion du tau hadronique) ;
- **JES** (`jes=f`) : multiplie le pt des jets ;
- **soft MET** (`soft_met=g`) : ajoute un terme mou (bruit) sur la MET.

On crée un jeu **nominal** (aucun biais) et les variations **±3 %** (TES, JES) ou **3 GeV** (soft MET). Le nominal passe lui aussi par `systematics()` pour une comparaison équitable.

In [ ]:
from HiggsML.systematics import systematics

SHIFT = 0.03       # ±3 %  (TES et JES)
SOFT_MET = 3.0     # GeV : terme mou ajouté à la MET

nominal  = systematics(df, tes=1.0)                 # référence (postprocess seul)

tes_up   = systematics(df, tes=1.0 + SHIFT)
tes_down = systematics(df, tes=1.0 - SHIFT)

jes_up   = systematics(df, jes=1.0 + SHIFT)
jes_down = systematics(df, jes=1.0 - SHIFT)

softmet  = systematics(df, soft_met=SOFT_MET)       # un seul sens : pas de bruit négatif sur la MET

print("événements chargés (nominal) :", len(nominal))

## 3. Listes de variables et fonctions d'aide

On masque la sentinelle `-25` (variables de jets non définies), on choisit des bins communs (entiers pour les comptages, intervalle robuste sinon), et on définit la fonction de tracé et la métrique de sensibilité.

In [ ]:
NON_FEATURES = {"weights", "labels", "detailed_labels"}
pri_features = [c for c in df.columns if c.startswith("PRI_") and c not in NON_FEATURES]
der_features = [c for c in df.columns if c.startswith("DER_") and c not in NON_FEATURES]
SENTINEL = -25.0


def _xlabel(col):
    """Unité physique sur l'axe x quand elle est connue."""
    low = col.lower()
    if "centrality" in low or "ratio" in low or "eta" in low:
        unit = ""
    elif "phi" in low:
        unit = "rad"
    elif "n_jets" in low:
        unit = ""
    elif ("pt" in low) or ("mass" in low) or ("met" in low):
        unit = "GeV"
    else:
        unit = ""
    return f"{col} [{unit}]" if unit else col


def _bins_for(col, n_bins=40, q=(0.5, 99.5)):
    """Bins communs, calculés sur le nominal (entiers si comptage, sinon intervalle robuste)."""
    x = nominal[col].to_numpy()
    xd = x[x != SENTINEL]
    uniq = np.unique(xd)
    if uniq.size <= 15 and np.allclose(uniq, np.round(uniq)):
        return np.arange(uniq.min() - 0.5, uniq.max() + 1.5, 1.0)
    lo, hi = np.percentile(xd, q)
    if lo == hi:
        lo, hi = xd.min(), xd.max()
    return np.linspace(lo, hi, n_bins + 1)


def _prob_hist(dset, col, bins):
    """Histogramme pondéré, normalisé en probabilité (somme = 1), sentinelle masquée."""
    x = dset[col].to_numpy()
    w = dset["weights"].to_numpy()
    d = x != SENTINEL
    h, _ = np.histogram(x[d], bins=bins, weights=w[d])
    s = h.sum()
    return h / s if s > 0 else h


def tv_distance(col, shifted):
    """Distance de variation totale (0–1) entre nominal et shifté : 0.5 * somme|p-q|."""
    bins = _bins_for(col)
    return 0.5 * np.abs(_prob_hist(nominal, col, bins) - _prob_hist(shifted, col, bins)).sum()


def _wmean(dset, col):
    x = dset[col].to_numpy()
    w = dset["weights"].to_numpy()
    d = x != SENTINEL
    return np.average(x[d], weights=w[d])


def _wstd(dset, col):
    """Écart-type pondéré (sentinelle masquée)."""
    x = dset[col].to_numpy()
    w = dset["weights"].to_numpy()
    d = x != SENTINEL
    xd, wd = x[d], w[d]
    m = np.average(xd, weights=wd)
    return np.sqrt(np.average((xd - m) ** 2, weights=wd))


def plot_syst_grid(features, suptitle, n_cols=4):
    """Grille : distribution nominale vs TES ±3 % (normalisée, pondérée), une case par feature."""
    n = len(features)
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.6 * n_cols, 3.4 * n_rows))
    axes = np.atleast_1d(axes).ravel()
    variations = [(tes_up, "#1f4fd8", "-"), (tes_down, "#d62728", "--")]
    for i, col in enumerate(features):
        ax = axes[i]
        bins = _bins_for(col)
        xn = nominal[col].to_numpy()
        wn = nominal["weights"].to_numpy()
        dn = xn != SENTINEL
        ax.hist(xn[dn], bins=bins, weights=wn[dn], density=True,
                histtype="stepfilled", color="0.75", alpha=0.7)
        for dset, color, ls in variations:
            x = dset[col].to_numpy()
            w = dset["weights"].to_numpy()
            d = x != SENTINEL
            ax.hist(x[d], bins=bins, weights=w[d], density=True,
                    histtype="step", color=color, lw=1.7, linestyle=ls)
        ax.set_title(col, fontsize=10)
        ax.set_xlabel(_xlabel(col), fontsize=9)
        ax.set_ylabel("densité", fontsize=8)
        ax.tick_params(labelsize=8)
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])
    handles = [Patch(facecolor="0.75", alpha=0.7, label="nominal (TES = 1.00)"),
               Line2D([0], [0], color="#1f4fd8", lw=1.7, label="TES +3 %"),
               Line2D([0], [0], color="#d62728", lw=1.7, ls="--", label="TES −3 %")]
    fig.legend(handles=handles, loc="upper center", ncol=3, fontsize=12, bbox_to_anchor=(0.5, 1.0))
    fig.suptitle(suptitle, fontsize=15, fontweight="bold", y=1.02)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    return fig

## 4. Superposition par feature — variables PRImaires (illustration TES)

In [ ]:
plot_syst_grid(pri_features, "Variables PRImaires — nominal vs TES ±3 %")
plt.show()

## 5. Superposition par feature — variables DERivées (illustration TES)

In [ ]:
plot_syst_grid(der_features, "Variables DERivées — nominal vs TES ±3 %")
plt.show()

## 6. Classement de sensibilité — un graphe en bâtons par biais

Pour chaque variable, un **score unique** : la **distance de variation totale** (TV) entre l'histogramme nominal et l'histogramme décalé (0 = aucun changement, 1 = distributions disjointes). Pour les biais à deux sens (TES, JES) on prend le maximum entre +3 % et −3 %. On ajoute le décalage de la **moyenne pondérée en fraction de l'écart-type** (`Δmoy/σ`), robuste même pour les variables centrées en 0.

On applique **exactement le même calcul** aux trois biais : **TES**, **JES**, **soft MET**.

In [ ]:
def classement(variations):
    """Tableau de sensibilité (TV + Δmoy/σ) d'un biais, trié du plus au moins sensible.
    `variations` = liste des jeux décalés à comparer au nominal (2 pour TES/JES, 1 pour soft MET)."""
    familles, scores, decalages = [], [], []
    for col in pri_features + der_features:
        # sensibilité = la plus grande variation parmi les jeux décalés
        score = max(tv_distance(col, v) for v in variations)
        # décalage de la moyenne (1er jeu décalé), en fraction de l'écart-type
        decalage = (_wmean(variations[0], col) - _wmean(nominal, col)) / _wstd(nominal, col)
        familles.append("PRI" if col.startswith("PRI_") else "DER")
        scores.append(score)
        decalages.append(decalage)

    ranking = pd.DataFrame({
        "feature": pri_features + der_features,
        "famille": familles,
        "TV": scores,
        "Δmoy/σ": decalages,
    })
    return ranking.sort_values("TV", ascending=False).reset_index(drop=True)


def trace_classement(ranking, titre):
    """Graphe en bâtons horizontal du score TV, trié, couleur par famille PRI/DER."""
    fig, ax = plt.subplots(figsize=(9, 9))
    couleurs = ["#1f77b4" if f == "PRI" else "#ff7f0e" for f in ranking["famille"]]
    ax.barh(ranking["feature"], ranking["TV"], color=couleurs)
    ax.invert_yaxis()
    ax.set_xlabel("Sensibilité — distance de variation totale (0 = insensible, 1 = max)")
    ax.set_title(titre, fontweight="bold")
    ax.legend(handles=[Patch(color="#1f77b4", label="PRI (primaire)"),
                       Patch(color="#ff7f0e", label="DER (dérivée)")])
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

### 6.1 — TES (tau energy scale)

In [ ]:
ranking_tes = classement([tes_up, tes_down])
trace_classement(ranking_tes, "Impact du biais TES (±3 %) sur chaque feature, trié")
ranking_tes.round(4)

### 6.2 — JES (jet energy scale)

In [ ]:
ranking_jes = classement([jes_up, jes_down])
trace_classement(ranking_jes, "Impact du biais JES (±3 %) sur chaque feature, trié")
ranking_jes.round(4)

### 6.3 — soft MET (terme mou sur la MET, 3 GeV)

In [ ]:
ranking_softmet = classement([softmet])
trace_classement(ranking_softmet, "Impact du biais soft MET (3 GeV) sur chaque feature, trié")
ranking_softmet.round(4)

## Lecture / pistes pour le WG FA

Chaque biais touche surtout les variables qui en dépendent physiquement :

- **TES** → `PRI_had_pt` (touché directement), puis la MET et les masses/sommes dérivées (`DER_mass_vis`, `DER_mass_transverse_met_lep`, `DER_sum_pt`…).
- **JES** → les variables de **jets** (`PRI_jet_*_pt`, `PRI_jet_all_pt`) et di-jet (`DER_*_jet_jet`, `DER_sum_pt`, `DER_pt_h`…).
- **soft MET** → la **MET** et ce qui en dépend (`PRI_met`, `DER_mass_transverse_met_lep`, `DER_met_phi_centrality`…). Comme c'est un *bruit* (élargissement) plus qu'un décalage, le score TV peut être notable alors que `Δmoy/σ` reste ≈ 0.

**Conséquence FA** : les variables en haut de **chaque** classement portent l'incertitude du biais correspondant ; c'est là qu'un systématique se traduira en incertitude sur μ. À privilégier/combiner avec prudence, et à croiser avec le WG SYST.